# DS4400 HW2

In [63]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## Problem 2

### Part A

In [12]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
#train_df.columns

In [13]:
non_numeric_cols = train_df.select_dtypes(include=['object']).columns.tolist()
columns_to_remove = ['ID'] + non_numeric_cols

In [14]:
train_numeric = train_df.drop(columns=columns_to_remove)
test_numeric = test_df.drop(columns=columns_to_remove)

In [15]:
print("Dropped columns:")
columns_to_remove

Dropped columns:


['ID',
 'Gender',
 'Occupation',
 'BMI_Category',
 'Blood_Pressure',
 'Sleep_Disorder']

In [18]:
feature_cols = [c for c in train_numeric.columns if c != target]

### Part B

In [19]:
target = 'Sleep_Quality'

In [20]:
X_train = train_numeric[feature_cols]
X_test = test_numeric[feature_cols]

y_train = train_numeric[target]
y_test = test_numeric[target]

In [21]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [22]:
print(f"Intercept: {model.intercept_:.6f}")

Intercept: 4.614076


In [23]:
print("Coefficients:")
for name, coef in zip(feature_cols, model.coef_):
    print(f"{name}: {coef:.6f}")

Coefficients:
Age: 0.013840
Sleep_Duration: 0.663838
Activity_Level: -0.000594
Stress_Level: -0.322086
Heart_Rate: -0.021018
Daily_Steps: 0.000092


In [24]:
y_train_pred = model.predict(X_train)

train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

In [25]:
print(f"Training MSE: {train_mse:.6f}")
print(f"Training R squared : {train_r2:.6f}")

Training MSE: 0.131003
Training R squared : 0.906411


### Part C

In [26]:
y_test_pred = model.predict(X_test)

test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

In [27]:
print(f"Testing MSE: {test_mse:.6f}")
print(f"Testing R squared : {test_r2:.6f}")

Testing MSE: 0.129190
Testing R squared : 0.912704


### Part D

The two features I chose are gender and BMI. I chose these two variables because gender and BMI are both common health-related factors that are likely to impact sleep quality. For gender, I encoded it using 0 and 1 since it can be considered a binary categorical variable (Female=0, Male=1). For BMI, since the categories have a natural order (Normal weight -> Overweight -> Obese), ordinal encoding can preserve this progression for the model (Normal/Normal Weight=0, Overweight=1, Obese=2).

In [28]:
# encoding gender: Female=0, Male=1
train_df['Gender_encoded'] = (train_df['Gender'] == 'Male').astype(int)
test_df['Gender_encoded'] = (test_df['Gender'] == 'Male').astype(int)

In [29]:
# to double check categories for BMI:
#print("BMI Categories:", train_df['BMI_Category'].unique()) 

In [30]:
# encoding BMI: Normal/Normal Weight=0, Overweight=1, Obese=2
bmi_order = {'Normal': 0, 'Normal Weight': 0, 'Overweight': 1, 'Obese': 2}
train_df['BMI_encoded'] = train_df['BMI_Category'].map(bmi_order)
test_df['BMI_encoded'] = test_df['BMI_Category'].map(bmi_order)

In [31]:
# model w/ new features
feature_cols_d = feature_cols + ['Gender_encoded', 'BMI_encoded']

X_train_d = train_df[feature_cols_d]
y_train_d = train_df[target]
X_test_d = test_df[feature_cols_d]
y_test_d = test_df[target]

In [32]:
model_d = LinearRegression()
model_d.fit(X_train_d, y_train_d)

LinearRegression()

In [33]:
# training r squared and MSE
y_train_pred_d = model_d.predict(X_train_d)
train_mse_d = mean_squared_error(y_train_d, y_train_pred_d)
train_r2_d = r2_score(y_train_d, y_train_pred_d)

In [34]:
# test r squared and MSE
y_test_pred_d = model_d.predict(X_test_d)
test_mse_d = mean_squared_error(y_test_d, y_test_pred_d)
test_r2_d = r2_score(y_test_d, y_test_pred_d)

In [35]:
print(f"Training MSE: {train_mse_d:.6f}   R squared: {train_r2_d:.6f}")
print(f"Testing  MSE: {test_mse_d:.6f}   R squared: {test_r2_d:.6f}")

Training MSE: 0.092476   R squared: 0.933935
Testing  MSE: 0.072431   R squared: 0.951057


### Part E

In [36]:
print(f"New Intercept: {model_d.intercept_:.6f}")

New Intercept: 4.406268


In [37]:
print("New Coefficients:")
for name, coef in zip(feature_cols_d, model_d.coef_):
    print(f"{name}: {coef:.6f}")

New Coefficients:
Age: 0.047325
Sleep_Duration: 0.210507
Activity_Level: 0.001779
Stress_Level: -0.477177
Heart_Rate: 0.023594
Daily_Steps: 0.000055
Gender_encoded: 0.280728
BMI_encoded: -0.680039


The features that contribute most to the linear regression model are BMI_encoded (coefficient = -0.680039) and Stress_Level (coefficient = -0.477177), both with negative relationships meaning higher BMI and higher stress are associated with lower sleep quality. Gender_encoded (0.280728) and Sleep_Duration (0.210507) also contribute meaningfully.

The model fits the data well. The initial model (Part b) had a training R squared of 0.906411 and testing R squared of 0.912704, meaning it explains about 91% of the variance in sleep quality. After adding the encoded features in Part (d), this improved to a training R squared of 0.933935 and testing R squared of 0.951057, explaining over 95% of the variance on the test set.

The model error is relatively small. The initial model had a training MSE of 0.131003 and testing MSE of 0.129190, while the improved model reduced these to 0.092476 and 0.072431 respectively. Since sleep quality is measured on a scale of roughly 4 to 9, an MSE of around 0.07 to 0.09 indicates the predictions are quite close to the actual values.

The training and testing MSE are close to each other in both models, and the testing MSE is actually slightly lower than the training MSE. This mean that the model is not overfitting.

## Problem 3

### Part A

The closed form solution for multiple linear regression is:

$$\mathbf{w} = (X^T X)^{-1} X^T y$$

where $X$ is the design matrix that includes a column of 1s added for the intercept and $y$ is the target vector. This is determined by setting the gradient of the MSE loss function to zero and solving for the weight vector. So, I had to add a column of 1s to $X$ so that the first element of $\mathbf{w}$ corresponds to the intercept (bias) term, and the remaining elements are the coefficients for each feature.

In [38]:
def train_closed_form(X, y):
    # add column of ones for intercept
    ones = np.ones((X.shape[0], 1))
    X_aug = np.hstack([ones, X])
    
    # closed form solution form
    w = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y
    return w

In this function, it takes in the feature matrix X and target vector y. It first adds a column of ones to the left side of X, which acts as the input for the intercept term. Then the weight vector is computed using the normal equation: $w = (X^TX)^{-1}X^Ty$. The result is a weight vector where the first element is the intercept and the remaining elements are the coefficients for each feature.

In [39]:
def predict(X, w):
    # add 1s
    ones = np.ones((X.shape[0], 1))
    
    # predict w/ learned weights
    X_aug = np.hstack([ones, X])
    return X_aug @ w

In this function, it takes in a new feature matrix X and the weight vector w from training. It then adds a column of ones to X, the same as I did during training, then it multiplies it by the weight vector ($X_w$) to get the predicted values.

In [51]:
m = train_closed_form(X_train, y_train)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


### Part B

In [52]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [53]:
def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

In [54]:
# closed form predictions
y_train_pred = predict(X_train, m)
y_test_pred = predict(X_test, m)

train_mse_cf = mse(y_train, y_train_pred)
train_r2_cf = r_squared(y_train, y_train_pred)
test_mse_cf = mse(y_test, y_test_pred)
test_r2_cf = r_squared(y_test, y_test_pred)

In [55]:
print("Closed-Form Implementation:")
print(f"Training MSE: {train_mse_cf:.6f} R squared: {train_r2_cf:.6f}")
print(f"Testing  MSE: {test_mse_cf:.6f}  R squared: {test_r2_cf:.6f}")

Closed-Form Implementation:
Training MSE: 0.131003 R squared: 0.906411
Testing  MSE: 0.129190  R squared: 0.912704


In [56]:
print("sklearn Implementation:")
print(f"Training MSE: {train_mse:.6f} R squared: {train_r2:.6f}")
print(f"Testing  MSE: {test_mse:.6f}  R squared: {test_r2:.6f}")

sklearn Implementation:
Training MSE: 0.131003 R squared: 0.906411
Testing  MSE: 0.129190  R squared: 0.912704


In [57]:
print(f"Differences:")
print(f"Training MSE diff: {abs(train_mse_cf - train_mse):.10f}")
print(f"Training R squared diff: {abs(train_r2_cf - train_r2):.10f}")
print(f"Testing MSE diff: {abs(test_mse_cf - test_mse):.10f}")
print(f"Testing R squared diff: {abs(test_r2_cf - test_r2):.10f}")

Differences:
Training MSE diff: 0.0000000000
Training R² diff: 0.0000000000
Testing MSE diff: 0.0000000000
Testing R² diff: 0.0000000000


## Problem 4

### Part A

In [58]:
def build_poly_features(X, p):
    # given a 1D feature array X and degree p, returns the polynomial feature matrix
    n = X.shape[0]
    X_poly = np.ones((n, p + 1))
    for i in range(1, p + 1):
        X_poly[:, i] = X ** i
        
    return X_poly

This function takes a 1D feature array X and a degree p, and constructs a matrix where the first column is all ones (for the intercept), the second column is X, the third is $X^2$, and so on up to $X^p$.

In [59]:
def train_poly(X, y, p):
    # train polynomial regresion of degree p using the closed-form solution, returns weight vector
    X_poly = build_poly_features(X, p)
    w = np.linalg.inv(X_poly.T @ X_poly) @ X_poly.T @ y
    
    return w

This function uses the matrix produced above and applies the same normal equation from before ($w = (X^TX)^{-1}X^Ty$) to find the weights.

In [60]:
def predict_poly(X, w, p):
# predict using polynomial regression weights
    X_poly = build_poly_features(X, p)
    
    return X_poly @ w

This function builds the same polynomial feature matrix for new data and multiplies it by the weight vector to get predictions.

### Part B

In [61]:
X_train_sd = train_df['Sleep_Duration'].values
y_train_sd = train_df['Sleep_Quality'].values
X_test_sd = test_df['Sleep_Duration'].values
y_test_sd = test_df['Sleep_Quality'].values

In [77]:
for p in range(1, 4):
    w = train_poly(X_train_sd, y_train_sd, p)
    
    y_train_pred = predict_poly(X_train_sd, w, p)
    y_test_pred = predict_poly(X_test_sd, w, p)
    
    tr_mse = mse(y_train_sd, y_train_pred)
    tr_r2 = r_squared(y_train_sd, y_train_pred)
    te_mse = mse(y_test_sd, y_test_pred)
    te_r2 = r_squared(y_test_sd, y_test_pred)
    
    print(f"P:{p}, Train MSE:{tr_mse}, Train R squared:{tr_r2}, Test MSE:{te_mse}, Test R squared:{te_r2}")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
P:1, Train MSE:0.30406594738900894, Train R squared:0.7827738332181491, Test MSE:0.3578027260210787, Test R squared:0.7582270819729235
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
P:2, Train MSE:0.29979643773152276, Train R squared:0.7858239913331821, Test MSE:0.3512321037034544, Test R squared:0.7626669546051189
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R)

In [76]:
# used chatgpt to help reprint the previous results to make it easier to read, tried a couple things but nothing got rid of MKL warnings
print(f"{'p':<5} {'Train MSE':<15} {'Train R squared':<15} {'Test MSE':<15} {'Test R squared':<15}")
print("-" * 65)
print(f"{'1':<5} {'0.304066':<15} {'0.782774':<15} {'0.357803':<15} {'0.758227':<15}")
print(f"{'2':<5} {'0.299796':<15} {'0.785824':<15} {'0.351232':<15} {'0.762667':<15}")
print(f"{'3':<5} {'0.292250':<15} {'0.791215':<15} {'0.331388':<15} {'0.776076':<15}")
print(f"{'4':<5} {'0.287645':<15} {'0.794505':<15} {'0.325631':<15} {'0.779966':<15}")
print(f"{'5':<5} {'0.281743':<15} {'0.798721':<15} {'0.319011':<15} {'0.784439':<15}")

p     Train MSE       Train R²        Test MSE        Test R²        
-----------------------------------------------------------------
1     0.304066        0.782774        0.357803        0.758227       
2     0.299796        0.785824        0.351232        0.762667       
3     0.292250        0.791215        0.331388        0.776076       


As $p$ increases from 1 to 3, training MSE consistently decreases and training $R^2$ consistently increases, meaning the model fits the training data better with higher degree polynomials. The same trend can be seen in the testing set, where test MSE decreases and test $R^2$ increases, showing that the higher degree polynomials also generalize better to unseen data. However, the improvements get smaller with each increase in $p$, suggesting diminishing returns from adding higher order terms.

### Part C

As $p$ increases, training MSE always decreases because a higher degree polynomial has more terms so it must have more flexibility to fit the training data closely. Testing MSE also decreases initially, but for very large values of p, testing MSE eventually starts to increase. This happens because the model begins to overfit, meaning it starts fitting the noise in the training data rather than the actual underlying relationship, which causes it to perform worse on unseen data. In our case with p from 1 to 3, both training and testing MSE are still decreasing, meaning the model has not yet started overfitting in this range.

## Problem 5

### Part A

In [82]:
def gradient_descent(X, y, lr=0.01, iterations=100):
    # adds a column of ones to X, returns the weight vector w where w[0] is the intercept
    n = X.shape[0]
    ones = np.ones((n, 1))
    X_aug = np.hstack([ones, X])
    w = np.zeros(X_aug.shape[1])

    # manually calculating gradient:
    for i in range(iterations):
        y_pred = X_aug @ w
        gradient = (2 / n) * X_aug.T @ (y_pred - y)
        w = w - lr * gradient

    return w

This function takes in a feature matrix X, target vector y, a learning rate, and a number of iterations. It first adds a column of ones to X for the intercept term, then initializes all weights to zero. In each iteration, it computes the predicted values ($Xw$), calculates the gradient of the MSE loss as: $(\frac{2}{n}) * X^T(Xw - y)$, and updates the weights by subtracting the learning rate times the gradient. After all iterations, the function returns the final weight vector. 

In [83]:
def predict_gd(X, w):
    # predict using the learned weights, adds a column of ones to X for the intercept
    ones = np.ones((X.shape[0], 1))
    X_aug = np.hstack([ones, X])
    
    return X_aug @ w

This function works the same way as before, adding a column of ones and computing the dot product with the weight vector.

In [84]:
# w/o normalization
w_no_norm = gradient_descent(X_train, y_train, lr=0.01, iterations=100)

print("Weights without normalization:", w_no_norm)

Weights without normalization: [nan nan nan nan nan nan nan]


/var/folders/pk/mxxxpcds47z3dkst84_c58pm0000gn/T/ipykernel_70751/404554697.py:11: RuntimeWarning: invalid value encountered in subtract
  w = w - lr * gradient


In [81]:
# w/ normalization
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

X_train_norm = (X_train - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

w_norm = gradient_descent(X_train_norm, y_train, lr=0.01, iterations=100)
print("Weights with normalization:", w_norm)

Weights with normalization: [ 6.39259387  0.15967289  0.47222391  0.11638101 -0.44222278 -0.23250331
  0.01057137]


### Part B

In [107]:
print(f"{'lr':<6} | {'iters':<7} | {'Train MSE':<15} | {'Train R squared':<15} | {'Test MSE':<15} | {'Test R squared':<15}")
print("-" * 88)

for lr in [0.01, 0.1, 0.5]:
    for iters in [10, 50, 100]:
        w = gradient_descent(X_train_norm, y_train, lr=lr, iterations=iters)

        y_train_pred = predict_gd(X_train_norm, w)
        y_test_pred = predict_gd(X_test_norm, w)

        tr_mse = mse(y_train, y_train_pred)
        tr_r2 = r_squared(y_train, y_train_pred)
        te_mse = mse(y_test, y_test_pred)
        te_r2 = r_squared(y_test, y_test_pred)

        print(f"{lr:<6} | {iters:<7} | {tr_mse:<7f}  | {tr_r2:<7f} | {te_mse:<7f} | {te_r2:<7f}")

lr     | iters   | Train MSE       | Train R squared | Test MSE        | Test R squared 
----------------------------------------------------------------------------------------
0.01   | 10      | 36.864028  | -25.335838 | 34.775535 | -22.498375
0.01   | 50      | 7.358074  | -4.256644 | 7.200612 | -3.865566
0.01   | 100     | 1.095162  | 0.217611 | 1.072661 | 0.275186
0.1    | 10      | 0.765662  | 0.453007 | 0.747508 | 0.494897
0.1    | 50      | 0.132574  | 0.905289 | 0.127631 | 0.913758
0.1    | 100     | 0.131299  | 0.906199 | 0.128287 | 0.913314
0.5    | 10      | 7781.520906  | -5558.155744 | 8359.526855 | -5647.663507
0.5    | 50      | 15378130817510494208.000000  | -10986210190396368896.000000 | 16518818100694843392.000000 | -11162024669225689088.000000
0.5    | 100     | 202633701299291921230215006945244348416.000000  | -144762485151781396930158596672152666112.000000 | 217664246231891695684157494248121303040.000000 | -147079147626584280774948128225738358784.000000


### Part C

With a small learning rate (0.01), the algorithm converges slowly and still has poor performance after 100 iterations ($R^2$ = 0.217), meaning it needs many more iterations to reach the optimal solution. With a moderate learning rate (0.1), the algorithm converges much faster, nearly matching the closed-form solution from Problem 3 by 50 iterations ($R^2$ = 0.905) and essentially reaching it by 100 iterations ($R^2$ = 0.906). With a large learning rate (0.5), the algorithm diverges completely, with MSE growing to extremely large numbers because the weight updates overshoot the minimum at each step. This shows that choosing the right learning rate is critical. If it's too small, convergence will be super slow but if it's too large, the algorithm will fail. With a balanced learning rate and enough iterations, gradient descent does converge to the same optimal solution as the closed-form method.